In [19]:
# cell 0
# Environment setup & repo root resolution + load secrets (credentials.json)
# IMPORTANT: DB identity comes from secrets; Azure SQL publishing is optional

import json
from pathlib import Path

# Resolve repo root even when notebook is run from /notebooks
NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

SECRETS_PATH = REPO_ROOT / "secrets" / "credentials.json"
if not SECRETS_PATH.exists():
    raise FileNotFoundError(
        f"Secrets file not found at: {SECRETS_PATH}\n"
        "Create it locally (and ensure it is in .gitignore)."
    )

with open(SECRETS_PATH, "r", encoding="utf-8") as f:
    SECRETS = json.load(f)

# Required blocks
BETFAIR = SECRETS["betfair"]
PATHS   = SECRETS["paths"]
USERCFG = SECRETS.get("user", {})

# Optional Azure SQL block (only required if enable_azure_sql=true)
AZSQL = SECRETS.get("azure_sql", None)

# Results directory (portable; configured in secrets)
RESULTS_CSV_DIR = Path(PATHS["results_csv_dir"])
RESULTS_CSV_DIR.mkdir(parents=True, exist_ok=True)

# DB write identity (comes from secrets; no hard-coded user identifiers in code)
DB_USER_ID = USERCFG.get("db_user_id", "").strip() or None

# Optional publishing toggle
ENABLE_AZURE_SQL = bool(USERCFG.get("enable_azure_sql", False))

# Safe-by-default: no writes unless explicitly flipped
DRY_RUN = True

print("Secrets loaded OK")
print("Repo root:", REPO_ROOT)
print("CSV output dir:", RESULTS_CSV_DIR)
print("Azure SQL enabled:", ENABLE_AZURE_SQL)
print("DB_USER_ID set:", bool(DB_USER_ID))
print("DRY_RUN:", DRY_RUN)


Secrets loaded OK
Repo root: c:\Users\Mark\OneDrive\Github\betfair-results-downloader
CSV output dir: C:\Users\Mark\OneDrive\BF Documentation\BF Results and Analysis\Results Database
Azure SQL enabled: False
DB_USER_ID set: True
DRY_RUN: True


In [ ]:
# cell 1
# Fetch Betfair cleared orders -> df_co (interactive login, no certs, paginated + retry/backoff)

import betfairlightweight
import pandas as pd
import datetime as dt
import json
import time
import pytz
from betfairlightweight.exceptions import APIError

# --- Credentials (loaded in Cell 0) ---
my_username = BETFAIR.get("username", "")
my_password = BETFAIR.get("password", "")
my_app_key  = BETFAIR.get("app_key", "")

print("Betfair username loaded:", my_username if my_username else "(blank)")
print("Betfair app key loaded:", "YES" if my_app_key else "NO")

# Keep this reasonable for testing; you can increase later
LOOKBACK_DAYS = 1
print("LOOKBACK_DAYS:", LOOKBACK_DAYS)

utc_now = dt.datetime.now(dt.timezone.utc)
from_dt = (utc_now - dt.timedelta(days=LOOKBACK_DAYS)).strftime("%Y-%m-%dT%H:%M:%SZ")
to_dt   = utc_now.strftime("%Y-%m-%dT%H:%M:%SZ")
print("Settled date range UTC:", from_dt, "->", to_dt)

settled_range = betfairlightweight.filters.time_range(from_=from_dt, to=to_dt)

# --- Connect (interactive login: NO certs) ---
trading = betfairlightweight.APIClient(
    username=my_username,
    password=my_password,
    app_key=my_app_key
)
trading.login_interactive()

def call_list_cleared_orders(from_record: int, record_count: int, max_retries: int = 5):
    """
    Call listClearedOrders with retry/backoff for Betfair TIMEOUT_ERROR.
    """
    for attempt in range(1, max_retries + 1):
        try:
            return trading.betting.list_cleared_orders(
                bet_status="SETTLED",
                settled_date_range=settled_range,
                from_record=from_record,
                record_count=record_count
            )
        except APIError as e:
            msg = str(e)
            is_timeout = ("TIMEOUT_ERROR" in msg) or ("ANGX-0010" in msg)
            if not is_timeout or attempt == max_retries:
                raise
            sleep_s = min(2 ** attempt, 20)  # capped exponential backoff
            print(f"Timeout from Betfair (attempt {attempt}/{max_retries}). Retrying in {sleep_s}s...")
            time.sleep(sleep_s)

# --- Paginate in smaller chunks to avoid timeouts ---
PAGE_SIZE = 200  # smaller = more reliable on slow APING days
indexrecord = 0
all_rows = []

while True:
    cleared_orders = call_list_cleared_orders(indexrecord, PAGE_SIZE)

    # Parse via JSON to be compatible across betfairlightweight response modes
    data = json.loads(cleared_orders.json())
    batch = data.get("clearedOrders", [])
    if not batch:
        break

    all_rows.extend(batch)
    indexrecord += PAGE_SIZE

df_co = pd.DataFrame(all_rows)

# Ensure required columns exist for downstream cells
required_cols = [
    "eventTypeId", "eventId", "marketId", "selectionId", "handicap", "betId",
    "placedDate", "persistenceType", "orderType", "side", "betOutcome",
    "priceRequested", "settledDate", "lastMatchedDate", "betCount",
    "priceMatched", "priceReduced", "sizeSettled", "profit",
    "customerOrderRef", "customerStrategyRef"
]
for c in required_cols:
    if c not in df_co.columns:
        df_co[c] = pd.NA
df_co = df_co[required_cols]

# Win column
def determine_win(row):
    if (row["side"] == "BACK" and row["betOutcome"] == "LOST") or (row["side"] == "LAY" and row["betOutcome"] == "WON"):
        return 0
    return 1

if not df_co.empty:
    df_co["Win"] = df_co.apply(determine_win, axis=1)
else:
    df_co["Win"] = pd.Series(dtype="int")

# placedDate to Australia/Sydney
df_co["placedDate"] = pd.to_datetime(df_co["placedDate"], utc=True, errors="coerce")
aet_zone = pytz.timezone("Australia/Sydney")
df_co["placedDate"] = df_co["placedDate"].dt.tz_convert(aet_zone)
df_co["placedDateOnly"] = df_co["placedDate"].dt.date
df_co["placedTimeOnly"] = df_co["placedDate"].dt.time

print("Rows:", len(df_co))
print("Columns:", df_co.columns.tolist())


In [ ]:
# cell 2
# Betfair credentials & API configuration (secrets)

print("Rows:", len(df_co))
print("Columns:", len(df_co.columns))
print(sorted(df_co.columns.tolist())[:60])   # first 60 alphabetically


In [ ]:
# cell 3
# Write snapshot CSV

from datetime import datetime

csv_out = RESULTS_CSV_DIR / f"cleared_orders_cleaned_{datetime.now():%Y-%m-%d}.csv"
df_co.to_csv(csv_out, index=False)
print(f"Wrote: {csv_out}")


In [ ]:
# cell 4
# CSV utilities: clean/merge/idempotent update (NO hard-coded paths)

import pandas as pd
import numpy as np
from pathlib import Path

def clean_and_remove_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean df_co-like cleared orders data and remove duplicates by betId.
    """
    df = df.copy()

    # Ensure betId is numeric and stable for de-duplication
    if "betId" in df.columns:
        df["betId"] = df["betId"].astype(np.int64)

    # Drop exact duplicate betId rows (keep last)
    if "betId" in df.columns:
        df = df.drop_duplicates(subset=["betId"], keep="last")

    return df


def update_csv_with_new_data(existing_csv_path: str, new_data_df: pd.DataFrame) -> None:
    """
    Idempotently update a CSV:
    - If file exists: load existing, concat new, dedupe by betId
    - If not: write cleaned new data
    Creates parent directory if missing.
    """
    path = Path(existing_csv_path)
    path.parent.mkdir(parents=True, exist_ok=True)

    if path.exists():
        existing_df = pd.read_csv(path)
        combined_df = pd.concat([existing_df, new_data_df], ignore_index=True)
        combined_df = clean_and_remove_duplicates(combined_df)
    else:
        combined_df = clean_and_remove_duplicates(new_data_df)

    combined_df.to_csv(path, index=False)


In [ ]:
# cell 5
# Define canonical CSV output paths (portable; uses secrets-defined directory)

from pathlib import Path
import datetime

# RESULTS_CSV_DIR is defined in Cell 0 from secrets/credentials.json
# e.g. PATHS["results_csv_dir"]
csv_output_dir = Path(RESULTS_CSV_DIR)
csv_output_dir.mkdir(parents=True, exist_ok=True)

# Canonical "always the same" file used for idempotent updates
csv_path = csv_output_dir / "cleared_orders_cleaned.csv"

# Optional dated snapshot for audit / rollback
today_str = datetime.date.today().isoformat()  # YYYY-MM-DD
csv_snapshot_path = csv_output_dir / f"cleared_orders_cleaned_{today_str}.csv"

print("CSV output dir:", str(csv_output_dir))
print("Canonical CSV path:", str(csv_path))
print("Snapshot CSV path:", str(csv_snapshot_path))
print("Canonical exists:", csv_path.exists())


In [ ]:
# cell 6
# Update / append closed orders CSV (idempotent), then write a dated snapshot

import pandas as pd

# Update canonical file idempotently
update_csv_with_new_data(str(csv_path), df_co)

# Snapshot the canonical file (so snapshot always reflects final canonical content)
df_canonical = pd.read_csv(csv_path)
df_canonical.to_csv(csv_snapshot_path, index=False)

print(f"Wrote canonical: {csv_path}")
print(f"Wrote snapshot : {csv_snapshot_path}")
print("Rows in canonical:", len(df_canonical))


In [ ]:
# cell 7
# Aggregate cleared orders -> market-level results (NO local DB / NO DSN)

import pandas as pd
from datetime import datetime

def log(msg: str):
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}")

log(f"df_co shape: {df_co.shape}")
log(f"df_co columns: {list(df_co.columns)}")

# Basic sanity checks
required = ["marketId", "profit", "betId", "placedDate"]
for c in required:
    if c not in df_co.columns:
        raise KeyError(f"Missing required column: {c}")

log(f"Nulls in betId: {df_co['betId'].isna().sum()}")
log(f"Nulls in marketId: {df_co['marketId'].isna().sum()}")
log(f"Nulls in profit: {df_co['profit'].isna().sum()}")
log(f"Nulls in placedDate: {df_co['placedDate'].isna().sum()}")

# Market-level aggregation: one row per MarketID
df_market_results = (
    df_co
    .groupby("marketId", as_index=False)
    .agg(
        Profit=("profit", "sum"),
        Bets=("betId", "count"),
        FirstPlaced=("placedDate", "min"),
        LastPlaced=("placedDate", "max"),
    )
)

log(f"df_market_results shape: {df_market_results.shape}")
display(df_market_results.head(10))


In [ ]:
# cell 8
# Build rows_to_write for Azure SQL (NO user id embedded) — round Profit to 2dp

from decimal import Decimal, ROUND_HALF_UP

assert "df_market_results" in globals(), "df_market_results not found. Run Cell 7 first."

def money2(x) -> Decimal:
    return Decimal(str(x)).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)

# Payload rows: (MarketID, Profit, Notes)
rows_to_write = []
for _, r in df_market_results.iterrows():
    market_id = Decimal(str(r["marketId"]))
    profit    = money2(r["Profit"])
    rows_to_write.append((market_id, profit, ""))

print("Prepared rows_to_write (market-level, no user id)")
print("Rows to write:", len(rows_to_write))
print("Sample (first 10):")
for row in rows_to_write[:10]:
    print(row)


In [ ]:
# cell 9
# Azure SQL pre-check (READ ONLY): row count + duplicates + profit variance

import pyodbc

assert ENABLE_AZURE_SQL is True, "Azure SQL is disabled. Set user.enable_azure_sql=true in secrets."
assert DB_USER_ID is not None, "DB user id missing. Set user.db_user_id in secrets."
assert AZSQL is not None, "azure_sql block missing in secrets."

az_conn_str = (
    f"DRIVER={{{AZSQL['driver']}}};"
    f"SERVER={AZSQL['server']},{AZSQL['port']};"
    f"DATABASE={AZSQL['database']};"
    f"UID={AZSQL['username']};"
    f"PWD={AZSQL['password']};"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "Connection Timeout=30;"
)

conn = None
cur = None

try:
    print("Testing Azure SQL connection (read-only)...")
    conn = pyodbc.connect(az_conn_str)
    cur = conn.cursor()
    cur.execute("SELECT 1;")
    print("Azure SQL connection OK.")

    cur.execute("SELECT COUNT(*) FROM dbo.MarketResults WHERE RTRIM(UserID) = ?;", (DB_USER_ID,))
    n = cur.fetchone()[0]
    print(f"Existing MarketResults rows for {DB_USER_ID}: {n}")

    sql = """
    SELECT
      RTRIM(UserID) AS UserID,
      MarketID,
      COUNT(*) AS cnt,
      MIN(Profit) AS min_profit,
      MAX(Profit) AS max_profit
    FROM dbo.MarketResults
    WHERE RTRIM(UserID) = ?
    GROUP BY RTRIM(UserID), MarketID
    HAVING COUNT(*) > 1
    ORDER BY cnt DESC;
    """
    cur.execute(sql, (DB_USER_ID,))
    rows = cur.fetchall()

    if not rows:
        print("No duplicates found for (UserID, MarketID).")
    else:
        print(f"WARNING: duplicates found for (UserID, MarketID): {len(rows)} groups. Showing up to 20:")
        for r in rows[:20]:
            print(r)

        conflicting = sum(1 for r in rows if r[3] != r[4])
        print(f"Conflicting duplicate groups (min_profit != max_profit): {conflicting} of {len(rows)}")

finally:
    if cur is not None:
        cur.close()
    if conn is not None:
        conn.close()


In [ ]:
# cell 10
# Azure SQL: post-write row count check (read-only, pyodbc)

import pyodbc

assert ENABLE_AZURE_SQL is True, "Azure SQL is disabled. Set user.enable_azure_sql=true in secrets."
assert DB_USER_ID is not None, "DB user id missing. Set user.db_user_id in secrets."
assert AZSQL is not None, "azure_sql block missing in secrets."

az_conn_str = (
    f"DRIVER={{{AZSQL['driver']}}};"
    f"SERVER={AZSQL['server']},{AZSQL['port']};"
    f"DATABASE={AZSQL['database']};"
    f"UID={AZSQL['username']};"
    f"PWD={AZSQL['password']};"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "Connection Timeout=30;"
)

conn = None
cur = None

try:
    print("Testing Azure SQL connection (read-only)...")
    conn = pyodbc.connect(az_conn_str)
    cur = conn.cursor()

    cur.execute("SELECT COUNT(*) FROM dbo.MarketResults WHERE RTRIM(UserID) = ?;", (DB_USER_ID,))
    n = cur.fetchone()[0]
    print(f"Azure: rows for DB_USER_ID={DB_USER_ID!r}: {n}")

finally:
    if cur is not None:
        cur.close()
    if conn is not None:
        conn.close()


In [ ]:
# cell 11
# Azure SQL: rebuild MarketResults for DB_USER_ID (DELETE then INSERT)

import pyodbc

assert ENABLE_AZURE_SQL is True, "Azure SQL is disabled. Set user.enable_azure_sql=true in secrets."
assert DB_USER_ID is not None, "DB user id missing. Set user.db_user_id in secrets."
assert AZSQL is not None, "azure_sql block missing in secrets."
assert DRY_RUN is False, "DRY_RUN must be False to write to Azure SQL."
assert "rows_to_write" in globals() and len(rows_to_write) > 0, "rows_to_write not prepared. Run Cell 8."

az_conn_str = (
    f"DRIVER={{{AZSQL['driver']}}};"
    f"SERVER={AZSQL['server']},{AZSQL['port']};"
    f"DATABASE={AZSQL['database']};"
    f"UID={AZSQL['username']};"
    f"PWD={AZSQL['password']};"
    "Encrypt=yes;"
    "TrustServerCertificate=no;"
    "Connection Timeout=30;"
)

conn = None
cur = None

try:
    print(f"Rebuilding dbo.MarketResults for DB_USER_ID={DB_USER_ID!r} ...")
    conn = pyodbc.connect(az_conn_str)
    conn.autocommit = False
    cur = conn.cursor()

    cur.execute("DELETE FROM dbo.MarketResults WHERE RTRIM(UserID) = ?;", (DB_USER_ID,))
    print("Deleted rows:", cur.rowcount)

    # Add DB_USER_ID at write time (not stored in rows_to_write)
    rows_for_db = [(DB_USER_ID, m, p, n) for (m, p, n) in rows_to_write]

    cur.fast_executemany = True
    cur.executemany(
        "INSERT INTO dbo.MarketResults (UserID, MarketID, Profit, Notes) VALUES (?, ?, ?, ?);",
        rows_for_db
    )

    conn.commit()
    print("✅ Commit complete.")

finally:
    if cur is not None:
        cur.close()
    if conn is not None:
        conn.close()
